In [36]:
"""
Track B — Final ID Resolution Pipeline
========================================
Files: depmap_expr, hpa_rna, hpa_desc, geo_expr

Output shape (per work_with_lookup.md):
    ensg_id | model_id | [dataset value columns]

This script incorporates every finding from the Track B investigation:
  - Bracket-collision disambiguation in Cellosaurus name matching
    (e.g. "BJ [human fibroblast]" vs "BJ [human B-cell]" vs
    "BJ [human pancreatic adenocarcinoma]" all collapsing to "bj")
  - "none"-as-string null detection in geo_info.cellosaurus_id
    (a plain .isna() check misses these — they are NOT true nulls)
  - cvcl_7082 exclusion (478 GEO samples with an unverifiable CVCL —
    no supporting name/study/matching_type evidence)
  - depmap_expr dual-profile deduplication (16 cell lines with two
    PR- IDs — keep highest-coverage profile)
  - Three-way deprecation check against sample_info.cellosaurus_issues,
    cross-referenced with signatures (NOT a blanket drop):
        clean          -> not flagged at all
        disputed       -> flagged removed, but still active in signatures
                          (do not drop -- flag for Track A)
        confirmed_gone  -> flagged removed AND absent from signatures
                          (safe to exclude)
  - hpa_desc promoted to Tier 1 for hpa_rna cell line resolution
    (confirmed 100% same-source name overlap with hpa_rna)
  - Cellosaurus cross-reference recovery for unresolved depmap_expr
    cell lines (depmap; ach-XXXXXX parsed from cross-references)
  - GEO ambiguous-CVCL disambiguation via geo_info's own "title" field
  - gene_lookup completeness check (flags real genes the lookup
    table is missing, rather than silently filtering them out)

Every unresolved / excluded / disputed row is logged with a reason.
Nothing is silently dropped.
"""

import os
import re
import pandas as pd

REF = "../../reference/"
DATA = "../../data/parquet/data_clean/"
LOGS = "logs/"
OUT = "resolved/"

os.makedirs(LOGS, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

# ============================================================
# STEP 0 — LOAD EVERYTHING
# ============================================================

gene_lookup      = pd.read_parquet(REF + "gene_lookup.parquet")
cell_line_lookup = pd.read_parquet(REF + "cell_line_lookup.parquet")

sample_info     = pd.read_parquet(DATA + "sample_info_clean.parquet")
depmap_profiles = pd.read_parquet(DATA + "depmap_profiles_clean.parquet")
signatures      = pd.read_parquet(DATA + "signatures_clean.parquet")
cellosaurus     = pd.read_parquet(DATA + "cellosaurus_clean.parquet")
geo_info        = pd.read_parquet(DATA + "geo_info_clean.parquet")
hpa_desc_raw    = pd.read_parquet(DATA + "hpa_desc_clean.parquet")

depmap_raw = pd.read_parquet(DATA + "depmap_expr_clean.parquet")
hpa_raw    = pd.read_parquet(DATA + "hpa_rna_clean.parquet")
geo_raw    = pd.read_parquet(DATA + "geo_expr_clean.parquet")

# Bring shared lookup tables into the same lowercase convention
# every other cleaned file in this project already follows
for col in gene_lookup.select_dtypes(include="object").columns:
    gene_lookup[col] = gene_lookup[col].str.lower()

for col in cell_line_lookup.select_dtypes(include="object").columns:
    cell_line_lookup[col] = cell_line_lookup[col].str.lower()

# Sanity check: do native IDs actually overlap with the lookup table
# at all, before running the full resolution chain?
native_ach_sample = set(depmap_profiles["modelid"].dropna().head(20))
overlap = native_ach_sample & set(cell_line_lookup["model_id"])
print(f"Quick overlap sanity check: {len(overlap)}/20 — should be close to 20")
if len(overlap) == 0:
    raise ValueError("Zero overlap between native IDs and cell_line_lookup — "
                      "check for a case mismatch before proceeding.")




# ============================================================
# STEP 1 — VALIDATE THE LOOKUP TABLES THEMSELVES
# ============================================================

print("=" * 60)
print("STEP 1 — VALIDATING SHARED LOOKUP TABLES")
print("=" * 60)

assert gene_lookup["ensg_id"].is_unique, "Duplicate ensg_id in gene_lookup!"
assert cell_line_lookup["model_id"].is_unique, "Duplicate model_id in cell_line_lookup!"
print(f"gene_lookup: {len(gene_lookup)} rows, unique ensg_id confirmed")
print(f"cell_line_lookup: {len(cell_line_lookup)} rows, unique model_id confirmed")

# Check 1a — duplicate CVCL accessions claimed by multiple model_ids
# (the CTV-1 / RH-30 pattern)
cvcl_dupe_counts = cell_line_lookup["cvcl_accession"].value_counts()
cvcl_dupes = cvcl_dupe_counts[cvcl_dupe_counts > 1]
print(f"\nCVCL accessions claimed by >1 model_id in cell_line_lookup: {len(cvcl_dupes)}")
if len(cvcl_dupes) > 0:
    print(cell_line_lookup[cell_line_lookup["cvcl_accession"].isin(cvcl_dupes.index)][
        ["model_id", "cell_line_name", "cvcl_accession"]
    ].to_string())
    pd.DataFrame({"cvcl_accession": cvcl_dupes.index, "n_model_ids": cvcl_dupes.values}).to_csv(
        LOGS + "lookup_issue_duplicate_cvcl.csv", index=False
    )
    print("  -> logged to lookup_issue_duplicate_cvcl.csv -- flag to Track A")

# Check 1b — three-way deprecation classification, built once, reused everywhere
deprecated_flagged = set(sample_info[sample_info["cellosaurus_issues"].astype(str).str.contains(
    "removed from depmap", case=False, na=False
)]["depmap_id"])
deprecated_disputed = deprecated_flagged & set(signatures["modelid"])
deprecated_confirmed = deprecated_flagged - deprecated_disputed

print(f"\nDeprecation flags in sample_info: {len(deprecated_flagged)}")
print(f"  disputed (flagged removed, but still active in signatures): {len(deprecated_disputed)}")
print(f"  confirmed_gone (flagged removed, absent from signatures too): {len(deprecated_confirmed)}")

pd.DataFrame({"model_id": list(deprecated_disputed)}).to_csv(
    LOGS + "lookup_issue_disputed_deprecation.csv", index=False
)
print("  -> disputed IDs logged to lookup_issue_disputed_deprecation.csv -- flag to Track A")

# Check 1c — does cell_line_lookup itself already contain deprecated rows?
deprecated_in_lookup = cell_line_lookup["model_id"].isin(deprecated_flagged).sum()
print(f"\ncell_line_lookup rows resting on a flagged model_id: {deprecated_in_lookup}")

# Check 1d — gene_lookup completeness (run for real, not left as a TODO)
print(f"\ngene_lookup unique ensg_id: {gene_lookup['ensg_id'].nunique()}")


def check_gene_completeness(native_genes, dataset_name):
    native_genes = set(native_genes.dropna())
    missing = native_genes - set(gene_lookup["ensg_id"])
    print(f"  {dataset_name}: {len(native_genes)} native genes, "
          f"{len(missing)} missing from gene_lookup")
    if len(missing) > 0:
        pd.Series(sorted(missing)).to_csv(
            LOGS + f"lookup_issue_genes_missing_from_lookup_{dataset_name}.csv", index=False
        )
    return missing


# ============================================================
# SHARED HELPER FUNCTIONS
# ============================================================

def normalise_base(name):
    """Lowercase, strip bracketed content, strip hyphen/slash/underscore/+/space."""
    if pd.isna(name):
        return None
    name = str(name).lower()
    name = re.sub(r"\[.*?\]", "", name)
    name = re.sub(r"[-/_+\s]", "", name)
    return name.strip()


def get_bracket_content(name):
    """Extract content inside brackets -- the disambiguating signal for
    cases like 'BJ [human fibroblast]' vs 'BJ [human B-cell]'."""
    m = re.search(r"\[(.*?)\]", str(name))
    return m.group(1).lower().strip() if m else None


def classify_deprecation(model_id):
    """Returns one of: 'clean', 'disputed', 'confirmed_gone', or None
    (None means not flagged at all -- safe to use)."""
    if model_id in deprecated_confirmed:
        return "confirmed_gone"
    if model_id in deprecated_disputed:
        return "disputed"
    return None  # not deprecated at all


def resolve_gene_column(values, dataset_name):
    """Validate a column of already-bare-ENSG values against gene_lookup.
    Logs anything missing rather than silently filtering."""
    missing = check_gene_completeness(values, dataset_name)
    valid_ensg = set(gene_lookup["ensg_id"])
    mask = values.isin(valid_ensg)
    print(f"  {dataset_name} gene resolution: {mask.sum()}/{len(values)} resolved")
    return mask


# Cellosaurus cross-reference parser (used for depmap_expr orphan recovery)
def extract_depmap_xref(xref_string):
    m = re.search(r"depmap;\s*(ach-\d+)", str(xref_string))
    return m.group(1) if m else None


cellosaurus["depmap_xref"] = cellosaurus["cross-references"].apply(extract_depmap_xref)
cellosaurus["base_norm"] = cellosaurus["cellosaurus_cell_line_name"].apply(normalise_base)
cellosaurus["bracket_content"] = cellosaurus["cellosaurus_cell_line_name"].apply(get_bracket_content)

cell_line_lookup["name_norm"] = cell_line_lookup["cell_line_name"].apply(normalise_base)
cell_line_lookup["stripped_norm"] = cell_line_lookup["stripped_cell_line_name"].apply(normalise_base)


# ============================================================
# FILE 1 — depmap_expr
# ============================================================

print("\n" + "=" * 60)
print("FILE 1 — depmap_expr")
print("=" * 60)

# --- Gene resolution: extract bare ENSG from column headers ---
def extract_ensg(col_header):
    m = re.search(r"ensg\d+", str(col_header))
    return m.group(0) if m else None

col_to_ensg = {c: extract_ensg(c) for c in depmap_raw.columns}
unparseable_cols = [c for c, e in col_to_ensg.items() if e is None]
print(f"Unparseable gene columns: {len(unparseable_cols)}")
if unparseable_cols:
    pd.Series(unparseable_cols).to_csv(LOGS + "unmapped_genes_depmap_expr_headers.csv", index=False)

depmap = depmap_raw.rename(columns=col_to_ensg)
extracted_genes = pd.Series([e for e in col_to_ensg.values() if e is not None])
gene_mask_depmap = resolve_gene_column(extracted_genes, "depmap_expr")

# --- Cell line resolution: PR- -> depmap_profiles -> cell_line_lookup ---
rna_profiles = depmap_profiles[depmap_profiles["datatype"] == "rna"]
pr_to_ach = dict(zip(rna_profiles["profileid"], rna_profiles["modelid"]))

dx = pd.DataFrame({"profile_id": depmap.index})
dx["model_id"] = dx["profile_id"].map(pr_to_ach)

unbridged = dx[dx["model_id"].isna()]
if len(unbridged) > 0:
    unbridged.to_csv(LOGS + "unmapped_cells_depmap_expr_no_bridge.csv", index=False)
    print(f"PR- IDs with no depmap_profiles bridge at all: {len(unbridged)}")

dx = dx.dropna(subset=["model_id"])

# Dual-profile dedup -- keep highest-coverage profile per model_id
coverage = depmap.sum(axis=1)
dx["coverage"] = dx["profile_id"].map(coverage)
dual_profile_count = dx["model_id"].duplicated().sum()
print(f"Dual-profile cell lines found: {dual_profile_count}")
dx = (dx.sort_values("coverage", ascending=False)
      .drop_duplicates(subset="model_id", keep="first")
      .reset_index(drop=True))
print(f"After dual-profile dedup: {len(dx)} unique cell lines")

# Validate against cell_line_lookup, with three-way deprecation classification
dx = dx.merge(cell_line_lookup[["model_id", "cell_line_name", "cvcl_accession"]],
              on="model_id", how="left")

orphaned_mask = dx["cell_line_name"].isna()
orphaned_ids = set(dx.loc[orphaned_mask, "model_id"])
print(f"Orphaned against cell_line_lookup: {len(orphaned_ids)}")

# Recovery attempt 1 -- confirm existence via signatures (does not give identity)
in_signatures = orphaned_ids & set(signatures["modelid"])
print(f"  confirmed real via signatures (existence only): {len(in_signatures)}")

# Recovery attempt 2 -- Cellosaurus cross-reference (gives full identity)
recovered = cellosaurus[cellosaurus["depmap_xref"].isin(orphaned_ids)][
    ["depmap_xref", "cellosaurus_cell_line_name", "cellosaurus_accession"]
].rename(columns={"depmap_xref": "model_id",
                   "cellosaurus_cell_line_name": "cell_line_name",
                   "cellosaurus_accession": "cvcl_accession"})
print(f"  recovered with full identity via Cellosaurus xref: {len(recovered)}")

dx = dx[~orphaned_mask]  # drop the orphan placeholder rows
dx_recovered = recovered.copy()
dx_recovered["coverage"] = dx_recovered["model_id"].map(
    dict(zip(dx["model_id"], dx.get("coverage", pd.Series(dtype=float))))
)
dx_recovered["profile_id"] = None
dx = pd.concat([dx, dx_recovered], ignore_index=True)

still_orphaned = orphaned_ids - set(recovered["model_id"])
pd.DataFrame({"model_id": list(still_orphaned)}).to_csv(
    LOGS + "unmapped_cells_depmap_expr_orphans.csv", index=False
)
print(f"  still genuinely unresolved: {len(still_orphaned)} -> logged")

# Three-way deprecation classification on the RESOLVED rows
dx["deprecation_status"] = dx["model_id"].apply(classify_deprecation)
print(f"\nResolved rows by deprecation status:")
print(dx["deprecation_status"].value_counts(dropna=False))

dx_disputed = dx[dx["deprecation_status"] == "disputed"].copy()
dx_disputed.to_csv(LOGS + "disputed_deprecation_depmap_expr.csv", index=False)

dx_clean = dx[dx["deprecation_status"].isna()].copy()   # not flagged at all -- safest
dx_confirmed_gone = dx[dx["deprecation_status"] == "confirmed_gone"]
dx_confirmed_gone.to_csv(LOGS + "confirmed_deprecated_depmap_expr.csv", index=False)

depmap_crosswalk = dx_clean[["profile_id", "model_id", "cvcl_accession", "cell_line_name"]].copy()

total_check = len(depmap_crosswalk) + len(dx_disputed) + len(dx_confirmed_gone) + len(still_orphaned)
print(f"\nReconciliation: clean({len(depmap_crosswalk)}) + disputed({len(dx_disputed)}) "
      f"+ confirmed_gone({len(dx_confirmed_gone)}) + orphaned({len(still_orphaned)}) "
      f"= {total_check}")
print(f"  (NOTE: disputed rows are held out of the clean crosswalk by default --")
print(f"   include dx_disputed manually if Track A confirms signatures is authoritative)")

assert depmap_crosswalk["model_id"].is_unique, "Duplicate model_id in final depmap_expr crosswalk!"


# ============================================================
# FILE 2 — hpa_rna
# ============================================================

print("\n" + "=" * 60)
print("FILE 2 — hpa_rna")
print("=" * 60)

gene_mask_hpa = resolve_gene_column(hpa_raw["gene"], "hpa_rna")

hpa_desc = hpa_desc_raw.copy()
hpa_lines = hpa_raw[["cell line"]].drop_duplicates().reset_index(drop=True)

results = []
for name in hpa_lines["cell line"]:
    base = normalise_base(name)
    bracket = get_bracket_content(name)

    # TIER 1 -- hpa_desc same-source exact name match (confirmed 100% overlap)
    desc_match = hpa_desc[hpa_desc["cell line"] == name]
    if len(desc_match) >= 1 and pd.notna(desc_match.iloc[0]["cellosaurus id"]):
        cvcl = desc_match.iloc[0]["cellosaurus id"]
        row = cell_line_lookup[cell_line_lookup["rrid"] == cvcl]
        if len(row) == 1:
            status = classify_deprecation(row.iloc[0]["model_id"])
            if status != "confirmed_gone":
                results.append({"hpa_name": name, "model_id": row.iloc[0]["model_id"],
                                 "cvcl_id": cvcl, "deprecation_status": status,
                                 "resolution_method": "hpa_desc_tier1"})
                continue

    # TIER 2 -- exact name match
    row = cell_line_lookup[cell_line_lookup["name_norm"] == base]
    if len(row) == 1:
        status = classify_deprecation(row.iloc[0]["model_id"])
        if status != "confirmed_gone":
            results.append({"hpa_name": name, "model_id": row.iloc[0]["model_id"],
                             "cvcl_id": row.iloc[0]["rrid"], "deprecation_status": status,
                             "resolution_method": "exact_tier2"})
            continue

    # TIER 3 -- stripped name match
    row = cell_line_lookup[cell_line_lookup["stripped_norm"] == base]
    if len(row) == 1:
        status = classify_deprecation(row.iloc[0]["model_id"])
        if status != "confirmed_gone":
            results.append({"hpa_name": name, "model_id": row.iloc[0]["model_id"],
                             "cvcl_id": row.iloc[0]["rrid"], "deprecation_status": status,
                             "resolution_method": "stripped_tier3"})
            continue

    # TIER 4 -- Cellosaurus name fallback, bracket-disambiguation-aware
    candidates = cellosaurus[cellosaurus["base_norm"] == base]
    chosen = None
    if len(candidates) == 1:
        chosen = candidates.iloc[0]
    elif len(candidates) > 1 and bracket is not None:
        refined = candidates[candidates["bracket_content"] == bracket]
        if len(refined) == 1:
            chosen = refined.iloc[0]

    if chosen is not None:
        cvcl = chosen["cellosaurus_accession"]
        row = cell_line_lookup[cell_line_lookup["rrid"] == cvcl]
        if len(row) == 1:
            status = classify_deprecation(row.iloc[0]["model_id"])
            if status != "confirmed_gone":
                results.append({"hpa_name": name, "model_id": row.iloc[0]["model_id"],
                                 "cvcl_id": cvcl, "deprecation_status": status,
                                 "resolution_method": "cellosaurus_tier4"})
                continue
        results.append({"hpa_name": name, "model_id": None, "cvcl_id": cvcl,
                         "deprecation_status": None,
                         "resolution_method": "hpa_only_no_depmap_equivalent"})
        continue

    results.append({"hpa_name": name, "model_id": None, "cvcl_id": None,
                     "deprecation_status": None, "resolution_method": "unresolved"})

hpa_crosswalk_full = pd.DataFrame(results)
print(hpa_crosswalk_full["resolution_method"].value_counts(dropna=False))
print(f"Total: {len(hpa_crosswalk_full)} (must equal {hpa_raw['cell line'].nunique()})")
assert len(hpa_crosswalk_full) == hpa_raw["cell line"].nunique()

disputed_hpa = hpa_crosswalk_full[hpa_crosswalk_full["deprecation_status"] == "disputed"]
disputed_hpa.to_csv(LOGS + "disputed_deprecation_hpa_rna.csv", index=False)

no_equivalent_hpa = hpa_crosswalk_full[
    hpa_crosswalk_full["resolution_method"] == "hpa_only_no_depmap_equivalent"
]
no_equivalent_hpa.to_csv(LOGS + "no_depmap_equivalent_hpa_rna.csv", index=False)

unresolved_hpa = hpa_crosswalk_full[hpa_crosswalk_full["resolution_method"] == "unresolved"]
unresolved_hpa.to_csv(LOGS + "unmapped_cells_hpa_rna.csv", index=False)

hpa_crosswalk = hpa_crosswalk_full[
    hpa_crosswalk_full["model_id"].notna() & (hpa_crosswalk_full["deprecation_status"].isna())
].copy()

assert hpa_crosswalk["hpa_name"].is_unique


# ============================================================
# FILE 3 — hpa_desc (reuse hpa_rna's crosswalk -- same source, same names)
# ============================================================

print("\n" + "=" * 60)
print("FILE 3 — hpa_desc")
print("=" * 60)
print("No gene column. Reusing hpa_rna's resolved crosswalk directly")
print("(confirmed 100% same-source name overlap -- no re-resolution needed).")

hpa_desc_resolved = hpa_desc.merge(
    hpa_crosswalk_full[["hpa_name", "model_id", "cvcl_id", "resolution_method"]],
    left_on="cell line", right_on="hpa_name", how="left"
)
print(f"hpa_desc resolved: {hpa_desc_resolved['model_id'].notna().sum()} / {len(hpa_desc_resolved)}")


# ============================================================
# FILE 4 — geo_expr
# ============================================================

print("\n" + "=" * 60)
print("FILE 4 — geo_expr")
print("=" * 60)

gene_mask_geo = resolve_gene_column(geo_raw["gene"], "geo_expr")

# --- Exclusions BEFORE resolution: corrected null detection + cvcl_7082 ---
true_null_mask = (
    geo_info["cellosaurus_id"].isna() |
    (geo_info["cellosaurus_id"].astype(str).str.lower() == "none")
)
null_gsms = set(geo_info[true_null_mask]["geo_accession"])
cvcl_7082_gsms = set(geo_info[geo_info["cellosaurus_id"] == "cvcl_7082"]["geo_accession"])
bad_gsms = null_gsms | cvcl_7082_gsms

print(f"Null CVCL GSMs excluded (incl. 'none'-string fix): {len(null_gsms)}")
print(f"cvcl_7082 unverifiable-identity GSMs excluded: {len(cvcl_7082_gsms)}")

gsm_cols = [c for c in geo_raw.columns if c != "gene"]
usable_gsms = [c for c in gsm_cols if c not in bad_gsms]
print(f"Total GSM columns: {len(gsm_cols)}, usable after exclusions: {len(usable_gsms)}")

excluded_log = pd.DataFrame(
    [{"gsm": g, "reason": "null_cvcl"} for g in null_gsms] +
    [{"gsm": g, "reason": "cvcl_7082_unverified"} for g in cvcl_7082_gsms]
)
excluded_log.to_csv(LOGS + "excluded_cells_geo_expr.csv", index=False)

results = []
for gsm in usable_gsms:
    row = geo_info[geo_info["geo_accession"] == gsm]
    cvcl = row.iloc[0]["cellosaurus_id"]
    title = row.iloc[0]["title"]

    matches = cell_line_lookup[cell_line_lookup["rrid"] == cvcl]

    if len(matches) == 0:
        results.append({"gsm": gsm, "model_id": None, "cvcl_id": cvcl,
                         "deprecation_status": None, "resolution_method": "no_depmap_equivalent"})
        continue

    if len(matches) == 1:
        status = classify_deprecation(matches.iloc[0]["model_id"])
        if status == "confirmed_gone":
            results.append({"gsm": gsm, "model_id": None, "cvcl_id": cvcl,
                             "deprecation_status": status,
                             "resolution_method": "confirmed_deprecated"})
        else:
            results.append({"gsm": gsm, "model_id": matches.iloc[0]["model_id"], "cvcl_id": cvcl,
                             "deprecation_status": status, "resolution_method": "resolved"})
        continue

    # Multiple sample_info/lookup matches for one CVCL -- disambiguate via
    # GEO's own submitted title field (the CTV-1 / CTV-1-DM pattern)
    title_match = matches[matches["cell_line_name"].apply(
        lambda x: str(x) in str(title).lower()
    )]
    if len(title_match) == 1:
        status = classify_deprecation(title_match.iloc[0]["model_id"])
        if status == "confirmed_gone":
            results.append({"gsm": gsm, "model_id": None, "cvcl_id": cvcl,
                             "deprecation_status": status,
                             "resolution_method": "confirmed_deprecated"})
        else:
            results.append({"gsm": gsm, "model_id": title_match.iloc[0]["model_id"], "cvcl_id": cvcl,
                             "deprecation_status": status, "resolution_method": "resolved_via_title"})
        continue

    # Still ambiguous -- do not guess
    results.append({"gsm": gsm, "model_id": None, "cvcl_id": cvcl,
                     "deprecation_status": None, "resolution_method": "ambiguous_unresolved"})

geo_crosswalk_full = pd.DataFrame(results)
print(geo_crosswalk_full["resolution_method"].value_counts(dropna=False))

disputed_geo = geo_crosswalk_full[geo_crosswalk_full["deprecation_status"] == "disputed"]
disputed_geo.to_csv(LOGS + "disputed_deprecation_geo_expr.csv", index=False)

geo_no_equivalent_or_gone = geo_crosswalk_full[
    geo_crosswalk_full["resolution_method"].isin(["no_depmap_equivalent", "confirmed_deprecated"])
]
geo_no_equivalent_or_gone.to_csv(LOGS + "no_depmap_equivalent_geo_expr.csv", index=False)

geo_ambiguous = geo_crosswalk_full[geo_crosswalk_full["resolution_method"] == "ambiguous_unresolved"]
geo_ambiguous.to_csv(LOGS + "ambiguous_unresolved_geo_expr.csv", index=False)

geo_crosswalk = geo_crosswalk_full[
    geo_crosswalk_full["model_id"].notna() & (geo_crosswalk_full["deprecation_status"].isna())
].copy()

total_geo = (len(geo_crosswalk) + len(disputed_geo) + len(geo_no_equivalent_or_gone) +
             len(geo_ambiguous) + len(bad_gsms))
print(f"\nReconciliation: {len(geo_crosswalk)} clean + {len(disputed_geo)} disputed "
      f"+ {len(geo_no_equivalent_or_gone)} no-equiv/gone + {len(geo_ambiguous)} ambiguous "
      f"+ {len(bad_gsms)} pre-excluded = {total_geo}")
assert total_geo == len(gsm_cols), f"Reconciliation mismatch! {total_geo} != {len(gsm_cols)}"


# ============================================================
# FINAL ASSEMBLY
#
# Required shape: ensg_id | model_id | [ALL original dataset columns]
#
# ensg_id and model_id are ADDED as new columns alongside whatever
# the dataset already has -- they are never used to replace or
# collapse existing columns (gene name, tpm, ptpm, etc. are all
# kept exactly as they were).
#
# For files that are NATIVELY long format (hpa_rna, hpa_desc):
#   no melt needed -- just merge the resolved keys onto the
#   existing rows and keep every original column untouched.
#
# For files that are NATIVELY wide format (depmap_expr, geo_expr):
#   gene and/or cell line identity lives in column headers, not
#   in row values -- melting is structurally unavoidable here,
#   because there IS no other native column to preserve per
#   (gene, cell line) pair beyond the single expression value
#   and the original native ID (kept for audit-trail purposes,
#   the same source_native_id convention used elsewhere in this
#   project).
# ============================================================

print("\n" + "=" * 60)
print("FINAL ASSEMBLY")
print("=" * 60)

# --- depmap_expr (wide -> long is required; only one native value
#     column exists per (gene, cell line) pair, so we keep the
#     native PR- profile_id alongside it for traceability) ---
depmap_valid_cols = [c for c in depmap.columns if c in set(gene_lookup["ensg_id"])]
depmap_long = (
    depmap[depmap_valid_cols]
    .loc[depmap_crosswalk["profile_id"].dropna()]
    .reset_index()
    .rename(columns={"index": "profile_id"})
    .melt(id_vars="profile_id", var_name="ensg_id", value_name="tpm_log2")
    .merge(depmap_crosswalk[["profile_id", "model_id"]], on="profile_id", how="inner")
    [["ensg_id", "model_id", "profile_id", "tpm_log2"]]
)
depmap_long.to_parquet(OUT + "depmap_expr_resolved.parquet", index=False)
print(f"depmap_expr_resolved.parquet: {len(depmap_long):,} rows, "
      f"columns: {depmap_long.columns.tolist()}")

# --- hpa_rna (ALREADY long format -- no melt needed.
#     Keep gene, gene name, cell line, tpm, ptpm, ntpm exactly as
#     they are; just add ensg_id and model_id as new columns) ---
hpa_long = (
    hpa_raw[hpa_raw["gene"].isin(gene_lookup["ensg_id"])]
    .merge(hpa_crosswalk[["hpa_name", "model_id"]],
           left_on="cell line", right_on="hpa_name", how="inner")
    .assign(ensg_id=lambda d: d["gene"])  # already validated == golden key
)
hpa_long = hpa_long[
    ["ensg_id", "model_id"] +
    [c for c in hpa_raw.columns if c not in ("gene",)]  # keep gene name, cell line, tpm, ptpm, ntpm
]
hpa_long.to_parquet(OUT + "hpa_rna_resolved.parquet", index=False)
print(f"hpa_rna_resolved.parquet: {len(hpa_long):,} rows, "
      f"columns: {hpa_long.columns.tolist()}")

# --- hpa_desc (no gene dimension -- model_id only, per spec.
#     Keep ALL original metadata columns, just add model_id) ---
hpa_desc_out = hpa_desc_resolved[hpa_desc_resolved["model_id"].notna()].copy()
hpa_desc_original_cols = [c for c in hpa_desc_raw.columns]  # everything hpa_desc originally had
hpa_desc_out = hpa_desc_out[["model_id"] + hpa_desc_original_cols]
hpa_desc_out.to_parquet(OUT + "hpa_desc_resolved.parquet", index=False)
print(f"hpa_desc_resolved.parquet: {len(hpa_desc_out):,} rows, "
      f"columns: {hpa_desc_out.columns.tolist()}")

# --- geo_expr (wide -> long is required; only one native value
#     column exists per (gene, cell line) pair, so we keep the
#     native GSM accession alongside it for traceability) ---
geo_valid = geo_raw[geo_raw["gene"].isin(gene_lookup["ensg_id"])]
geo_usable_resolved_cols = ["gene"] + [
    g for g in geo_crosswalk["gsm"] if g in geo_valid.columns
]
geo_long = (
    geo_valid[geo_usable_resolved_cols]
    .melt(id_vars="gene", var_name="gsm", value_name="rma_intensity")
    .merge(geo_crosswalk[["gsm", "model_id"]], on="gsm", how="inner")
    .rename(columns={"gene": "ensg_id"})
    [["ensg_id", "model_id", "gsm", "rma_intensity"]]
)
geo_long.to_parquet(OUT + "geo_expr_resolved.parquet", index=False)
print(f"geo_expr_resolved.parquet: {len(geo_long):,} rows, "
      f"columns: {geo_long.columns.tolist()}")

print("\n" + "=" * 60)
print("DONE. All outputs in resolved/, all logs in logs/.")
print("Review logs/disputed_deprecation_*.csv before final handoff --")
print("these are HELD OUT pending Track A confirmation, not silently dropped.")
print("=" * 60)

Quick overlap sanity check: 20/20 — should be close to 20
STEP 1 — VALIDATING SHARED LOOKUP TABLES
gene_lookup: 19446 rows, unique ensg_id confirmed
cell_line_lookup: 1840 rows, unique model_id confirmed

CVCL accessions claimed by >1 model_id in cell_line_lookup: 4
        model_id cell_line_name cvcl_accession
1135  ach-000833          rh-30      cvcl_0041
1299  ach-001024          chl-1      cvcl_1122
1311  ach-001041       chl-1-dm      cvcl_1122
1361  ach-001189           None      cvcl_0041
1499  ach-001543         kosc-2      cvcl_1337
1550  ach-001737       ctv-1-dm      cvcl_1150
1711  ach-002222          ctv-1      cvcl_1150
1749  ach-002260           None      cvcl_1337
  -> logged to lookup_issue_duplicate_cvcl.csv -- flag to Track A

Deprecation flags in sample_info: 11
  disputed (flagged removed, but still active in signatures): 8
  confirmed_gone (flagged removed, absent from signatures too): 3
  -> disputed IDs logged to lookup_issue_disputed_deprecation.csv -- flag to

In [ ]:
# Check what's actually being dropped for depmap_expr specifically
missing_genes = check_gene_completeness(extracted_genes, "depmap_expr_audit")

# Cross-check: how many of these ARE present in hpa_rna or geo_expr
# (both already confirmed reliable, protein-coding-filtered sources)?
hpa_genes = set(hpa_raw["gene"].dropna())
geo_genes = set(geo_raw["gene"].dropna())

falsely_dropped = missing_genes & (hpa_genes | geo_genes)
print(f"Genes dropped from depmap_expr that ARE real per HPA/GEO: {len(falsely_dropped)}")

  depmap_expr_audit: 53961 native genes, 34578 missing from gene_lookup
Genes dropped from depmap_expr that ARE real per HPA/GEO: 4404


In [ ]:
# Does cell_line_lookup (1,840 rows) actually match sample_info's
# scope, or is it a DIFFERENT, possibly even older snapshot?
print(set(cell_line_lookup["model_id"]) == set(sample_info["depmap_id"]))
print(len(set(sample_info["depmap_id"]) - set(cell_line_lookup["model_id"])))
print(len(set(cell_line_lookup["model_id"]) - set(sample_info["depmap_id"])))

True
0
0


In [37]:
hpa_rna = pd.read_parquet(OUT + "hpa_rna_resolved.parquet")

In [38]:
hpa_rna.head(10)

,ensg_id,model_id,gene name,cell line,tpm,ptpm,ntpm
0,ensg00000000003,ach-001001,tspan6,143b,22.0,27.6,25.9
1,ensg00000000003,ach-000956,tspan6,22rv1,2.8,3.6,2.7
2,ensg00000000003,ach-000948,tspan6,23132/87,6.2,7.5,7.5
3,ensg00000000003,ach-000011,tspan6,253j,14.2,18.7,25.4
4,ensg00000000003,ach-000026,tspan6,253j-bv,13.0,17.1,18.5
5,ensg00000000003,ach-000323,tspan6,42-mg-ba,14.1,17.7,18.6
6,ensg00000000003,ach-000905,tspan6,5637,33.5,39.7,44.4
7,ensg00000000003,ach-000520,tspan6,59m,14.4,18.0,20.1
8,ensg00000000003,ach-000973,tspan6,639v,16.9,21.5,20.3
9,ensg00000000003,ach-000896,tspan6,647v,32.5,41.1,50.9


In [ ]:
hpa_raw.head(10)

,gene,gene name,cell line,tpm,ptpm,ntpm
0,ensg00000000003,tspan6,143b,22.0,27.6,25.9
1,ensg00000000003,tspan6,22rv1,2.8,3.6,2.7
2,ensg00000000003,tspan6,23132/87,6.2,7.5,7.5
3,ensg00000000003,tspan6,253j,14.2,18.7,25.4
4,ensg00000000003,tspan6,253j-bv,13.0,17.1,18.5
5,ensg00000000003,tspan6,42-mg-ba,14.1,17.7,18.6
6,ensg00000000003,tspan6,537-mel,9.8,11.9,13.7
7,ensg00000000003,tspan6,5637,33.5,39.7,44.4
8,ensg00000000003,tspan6,59m,14.4,18.0,20.1
9,ensg00000000003,tspan6,624-mel,11.6,14.6,15.7


In [ ]:
print(len(still_orphaned))
print(still_orphaned - in_signatures - set(recovered["model_id"]))

41
{'ach-003161', 'ach-003160', 'ach-003139', 'ach-003152', 'ach-003156', 'ach-003135', 'ach-003149', 'ach-003141', 'ach-003148', 'ach-003159', 'ach-003145', 'ach-003157', 'ach-003150', 'ach-003154', 'ach-003136', 'ach-003132', 'ach-003155', 'ach-003133', 'ach-003143', 'ach-003153', 'ach-003158', 'ach-003142', 'ach-003138', 'ach-003147', 'ach-003134'}


In [39]:
depmap = pd.read_parquet(OUT + "depmap_expr_resolved.parquet")
depmap.head(10)

,ensg_id,model_id,profile_id,tpm_log2
0,ensg00000000003,ach-000904,pr-wpmvt8,3.748461
1,ensg00000000003,ach-000703,pr-objc9j,4.450881
2,ensg00000000003,ach-001321,pr-v3ipaf,1.000000
3,ensg00000000003,ach-000382,pr-axh1q4,4.257765
4,ensg00000000003,ach-000514,pr-uqbh8b,4.386121
5,ensg00000000003,ach-000177,pr-vs2hio,3.919340
6,ensg00000000003,ach-000203,pr-fyk1ur,4.364572
7,ensg00000000003,ach-000610,pr-wy4kp0,4.179511
8,ensg00000000003,ach-000870,pr-qxoaxl,5.498570
9,ensg00000000003,ach-000594,pr-chmlzf,3.397803


In [50]:
depmap_raw.transpose().shape

(53961, 1495)

In [41]:
depmap.shape

(26322114, 4)

In [42]:
depmap['ensg_id'].nunique()

19383

In [43]:
depmap_raw.head()

,ensg00000000003,ensg00000000005,ensg00000000419,ensg00000000457,ensg00000000460,ensg00000000938,ensg00000000971,ensg00000001036,ensg00000001084,ensg00000001167,...,ensg00000288714,ensg00000288717,ensg00000288718,ensg00000288719,ensg00000288720,ensg00000288721,ensg00000288722,ensg00000288723,ensg00000288724,ensg00000288725
pr-adbjpg,4.331992,0.000000,7.364660,2.792855,4.471187,0.028569,1.226509,3.044394,6.500005,4.739848,...,0.000000,0.536053,0.000000,0.028569,0.176323,0.992768,2.797013,0.000000,0.0,0.000000
pr-i2azwg,4.567424,0.584963,7.106641,2.543496,3.504620,0.000000,0.189034,3.813525,4.221877,3.481557,...,0.000000,0.879706,0.000000,0.014355,0.014355,0.432959,2.972693,0.056584,0.0,0.070389
pr-5ekaac,3.150560,0.000000,7.379118,2.333424,4.228049,0.056584,1.310340,6.687201,3.682573,3.273516,...,0.028569,0.000000,0.084064,0.000000,0.097611,0.367371,1.695994,0.084064,0.0,0.000000
pr-i21681,5.085340,0.000000,7.154211,2.545968,3.084064,0.000000,5.868390,6.165309,4.489928,3.956986,...,0.000000,0.000000,0.070389,0.000000,0.176323,0.411426,3.921246,0.028569,0.0,0.000000
pr-i9drp1,6.729417,0.000000,6.537917,2.456806,3.867896,0.799087,7.208478,5.570159,7.127117,4.568032,...,0.000000,0.000000,0.201634,0.028569,0.137504,0.678072,4.418190,0.000000,0.0,0.000000


In [45]:
depmappmap_profile = pd.read_parquet(DATA + "depmap_profiles_clean.parquet")

In [46]:
depmappmap_profile.head()

,profileid,modelcondition,modelid,datatype,weskit
0,pr-00utu3,mc-001131-kkjv,ach-001131,wgs,none
1,pr-01r7om,mc-000957-yckn,ach-000957,rna,none
2,pr-02xmlg,mc-002785-qo9e,ach-002785,rna,none
3,pr-04vvbz,mc-001289-bpdi,ach-001289,wes,ice
4,pr-09gmei,mc-000520-yim7,ach-000520,rna,none


In [48]:
depmap_profiles.shape

(3830, 5)

In [49]:
depmap_profiles.nunique()

profileid         3830
modelcondition    2540
modelid           1822
datatype             3
weskit               3
dtype: int64

In [52]:
geo_expr = pd.read_parquet(OUT + "geo_expr_resolved.parquet")
geo_expr.head()

,ensg_id,model_id,gsm,rma_intensity
0,ensg00000000003,ach-000558,gsm101610,33.615700
1,ensg00000000005,ach-000558,gsm101610,40.925682
2,ensg00000000419,ach-000558,gsm101610,2182.281250
3,ensg00000000457,ach-000558,gsm101610,58.934814
4,ensg00000000460,ach-000558,gsm101610,136.418900


In [55]:
geo_raw.shape

(19914, 3268)

In [56]:
geo_raw.head()

,gene,gsm101610,gsm101615,gsm101616,gsm101667,gsm101668,gsm101671,gsm101672,gsm101673,gsm101674,...,gsm960289,gsm960290,gsm960291,gsm960292,gsm960293,gsm960294,gsm960295,gsm960296,gsm960297,gsm960298
0,ensg00000000003,33.615700,553.249756,540.452209,599.431152,625.242737,400.554657,412.995605,427.403900,461.123535,...,197.712616,387.427826,116.657890,104.889442,158.400604,736.519043,631.033142,791.054504,181.986893,353.206940
1,ensg00000000005,40.925682,31.327406,33.934967,34.213123,32.466286,36.243233,37.952511,34.644428,34.225410,...,4.392303,4.230473,4.651096,5.175624,5.540960,4.211264,4.159354,4.480873,4.630062,4.296047
2,ensg00000000419,2182.281250,3419.430420,3514.540039,2295.817383,2378.469727,2297.564697,2276.069092,2368.761475,2689.160156,...,936.201782,1162.099731,1249.029053,1354.346069,1159.747559,1561.643799,1547.089233,1551.400146,939.964661,1054.350952
3,ensg00000000457,58.934814,95.068222,94.900459,51.810070,52.327530,50.011375,53.793667,51.172344,52.677490,...,35.778347,57.736485,25.528204,24.593216,23.337006,47.793488,41.268002,64.042831,37.363773,61.027988
4,ensg00000000460,136.418900,257.929169,271.317230,162.090073,160.913849,206.788635,209.966019,134.272583,147.408554,...,14.693585,19.083935,114.056015,137.948593,149.641708,111.793892,98.536613,167.151016,12.268769,24.296593


In [57]:
geo_expr.shape

(36676472, 4)

In [58]:
geo_info = pd.read_parquet(DATA + "geo_info_clean.parquet")
geo_info.head()

,geo_accession,cel_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_institute,gse_id,gse_filename,cell_line,disease,origin,cellosaurus_id,cellline,matching_type,cell_line_trimmed
0,gsm101610,gsm101610,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_0131,a-172,cello geo gsm,none
1,gsm101615,gsm101615,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_0393,ln-229,cello geo gsm,none
2,gsm101616,gsm101616,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_0393,ln-229,cello geo gsm,none
3,gsm101667,gsm101667,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_1715,sw1088,cello geo gsm,none
4,gsm101668,gsm101668,none,none,none,none,none,NaN,none,none,...,none,none,none,none,none,none,cvcl_1715,sw1088,cello geo gsm,none
